# Project: Intelligent Retail (Edge Model)
### Datum: 2026-03-15
### Team: Michal Kakol

## 0. Setup:

In deze sectie laden we de benodigde libraries. We gebruiken PySpark voor de Cloud pipeline om de grote hoeveelheid data (Volume) en de snelheid (Velocity) aan te kunnen. Voor de Edge component gebruiken we lichtere libraries zoals PIL en Pandas, omdat de hardware op locatie vaak beperkte resources heeft.

In [ ]:
import logging
import os
import pandas as pd
from PIL import Image
import pickle
import hashlib
import json
from datetime import datetime
logging.basicConfig(level=logging.DEBUG)

### 0.1. Edge:

In [ ]:
from functions.edge import *

### 0.2. Cloud:

In [ ]:
from functions.cloud import *

## 1. Data Loading:

De data-ingestie is opgesplitst in twee stromen:
- Edge: Gericht op ongestructureerde beelddata (Variety).
- Cloud: Gericht op gestructureerde historische verkoopcijfers (Volume).
    We gebruiken een specifiek Schema voor Spark. Dit zorgt voor een betere datakwaliteit en optimaliseert het laadproces.

### 1.1. Edge:

In [ ]:
processed_images, labels = edge_load_process("../data/raw/image/frames", "../data/raw/image/labels.csv")

### 1.2. Cloud:

Voor de cloud data maken we gebruik van Spark SQL concepten. Het laden gebeurt Lazy. Dit betekent dat Spark pas een executieplan uitvoert op het moment dat er echt een actie, zoals een count, wordt aangeroepen. Dit bespaart rekencapaciteit.

In [ ]:
df_cloud = cloud_load_transform(spark, "../data/raw/text/train.csv")

## 2. Data Processing:

Tijdens de verwerking monitoren we de pipeline. Dit is nodig om afwijkingen of veranderingen in de data direct te kunnen detecteren.

### 2.1. Edge:

Om het model lichtgewicht te houden, worden de afbeeldingen lokaal geschaald. Dit is een vorm van Vertical Scaling, waarbij we de beschikbare resources op een node zo efficient mogelijk gebruiken.

In [ ]:
edge_stats = edge_save_monitor(processed_images, labels, "../data/processed/images.pkl", "../data/monitoring/edge_stats.json")

### 2.2. Cloud:

We gebruiken "coalesce(1)" voordat we de data opslaan als Parquet file. Hiermee voorkomen we "Shuffling" tussen verschillende nodes in het cluster. Ook lossen we hiermee het "small file problem" op, wat anders de training van het machine learning model zou vertragen.

In [ ]:
cloud_stats = cloud_save_monitor(df_cloud, "../data/processed/sales.parquet", "../data/monitoring/cloud_stats.json")

## 3. Pipeline:

In deze functies staat de volledige logica voor een productieomgeving.

### 3.1. Edge:

Deze pipeline wordt aangestuurd door Event-Based logica. Zodra er nieuwe camerabeelden beschikbaar zijn, wordt de verwerking gestart.

In [ ]:
edge_pipeline("../data/raw/image/frames", "../data/raw/image/labels.csv", "../data/processed/images.pkl")

### 3.1. Cloud:

De cloud-pipeline is gericht op Batch Processing van de verkoopdata. Het maakt gebruik van het vooraf gedefinieerde StructType schema om de data-integriteit automatisch te controleren.

In [ ]:
cloud_pipeline(spark, "../data/raw/text/train.csv", "../data/processed/sales.parquet")